In [24]:
import pdfplumber
import re

In [25]:
# load data
math_modules = pdfplumber.open("../data/math_modules.pdf")
print("Page count:", len(math_modules.pages))

Page count: 54


In [ ]:
# need pages at idx 3 to idx 16
p0 = math_modules.pages[3]
p0 = p0.extract_tables()
p0

[[['Empfohlene Teilnahmevoraussetzung(en) für das Modul',
   'Teilnahme am mathematischen Vorkurs wird dringend\nempfohlen.'],
  ['bzw. für einzelne Lehrveranstaltungen des Moduls', None],
  ['Unterrichtssprache(n) und Prüfungssprache(n)', 'deutsch'],
  ['Stellenwert der Modulnote in der Gesamtnote', '0%'],
  ['Häufigkeit des Angebots', 'jedes Semester'],
  ['Begründung der Anwesenheitspflicht', ''],
  ['',
   'Studiengangsbeauftragte(r) (siehe Institutsseite)\n(siehe Institutsseite)'],
  ['Modulbeauftragte oder Modulbeauftragter', None],
  ['', None],
  ['Verwendbarkeit des Moduls in anderen Studiengängen',
   'B. Ed. Mathematik, B. Sc. Informatik, B. Sc. Physik'],
  ['',
   'Die Übung hat einen LP mehr als im\nLehramtsstudiengang. Entsprechend erhalten die\nB.Sc. zusätzliche ergänzende Übungsaufgaben'],
  ['Sonstiges', None],
  ['', None]],
 [['Modul LAG2',
   'Lineare Algebra und Geometrie 2',
   None,
   None,
   None,
   None,
   '[Modul-Kennnummer ]',
   None],
  [None, None, Non

In [27]:
def remove_none(tables):
    return [
        [
            [cell for cell in row if cell is not None]
            for row in table
        ]
        for table in tables
    ]
def remove_n(tables):
    return [
        [
            [re.sub(r"\s+", " ", str(cell)) for cell in row if cell is not None] # type: ignore
            for row in table
        ]
        for table in tables
    ]
def clean(tables):
    tables = remove_none(tables)
    tables = remove_n(tables)
    return tables
p0 = clean(p0)
p0

[[['Empfohlene Teilnahmevoraussetzung(en) für das Modul',
   'Teilnahme am mathematischen Vorkurs wird dringend empfohlen.'],
  ['bzw. für einzelne Lehrveranstaltungen des Moduls'],
  ['Unterrichtssprache(n) und Prüfungssprache(n)', 'deutsch'],
  ['Stellenwert der Modulnote in der Gesamtnote', '0%'],
  ['Häufigkeit des Angebots', 'jedes Semester'],
  ['Begründung der Anwesenheitspflicht', ''],
  ['',
   'Studiengangsbeauftragte(r) (siehe Institutsseite) (siehe Institutsseite)'],
  ['Modulbeauftragte oder Modulbeauftragter'],
  [''],
  ['Verwendbarkeit des Moduls in anderen Studiengängen',
   'B. Ed. Mathematik, B. Sc. Informatik, B. Sc. Physik'],
  ['',
   'Die Übung hat einen LP mehr als im Lehramtsstudiengang. Entsprechend erhalten die B.Sc. zusätzliche ergänzende Übungsaufgaben'],
  ['Sonstiges'],
  ['']],
 [['Modul LAG2', 'Lineare Algebra und Geometrie 2', '[Modul-Kennnummer ]'],
  [''],
  ['', 'Linear Algebra and Geometry 2'],
  ['Pflicht- oder Wahlpflichtmodul', 'Pflichtmodul'],


In [28]:
STOPWORDS = ["in", "im", "am", "an", "zu", "LP", "für", "wird", "das", "des", "der", "die", "weil", "und", "oder"]

pattern = re.compile(r'([a-zA-ZäöüÄÖÜß]{2,})\s+([a-zäöüß]{1,2})(?=\s|$)')

def fix_broken_words(text: str) -> str:
    def replacer(match):
        left = match.group(1)
        right = match.group(2)

        # nichts tun, wenn right Stopword ist
        if right in STOPWORDS:
            return match.group(0)

        return left + right  # zusammenfügen

    # einmal durchgehen (kein while!)
    return pattern.sub(replacer, text)

    return text

def tables_to_text(tables):
    lines = []

    for table in tables:
        for row in table:
            # leere Zellen rausfiltern
            cells = [c for c in row if c]

            if not cells:
                continue

            # 1 Zelle → einfach übernehmen
            if len(cells) == 1:
                lines.append(cells[0])

            # mehrere Zellen → key: value
            else:
                lines.append(": ".join(cells))

    text = " ".join(lines)
    text = text.replace("\uf0b7", "-")
    return fix_broken_words(text)

p0 = tables_to_text(p0)
p0

'Empfohlene Teilnahmevoraussetzung(en) für das Modul: Teilnahme am mathematischen Vorkurs wird dringend empfohlen. bzw. für einzelne Lehrveranstaltungen des Moduls Unterrichtssprache(n) und Prüfungssprache(n): deutsch Stellenwert der Modulnote in der Gesamtnote: 0% Häufigkeit des Angebots: jedes Semester Begründung der Anwesenheitspflicht Studiengangsbeauftragte(r) (siehe Institutsseite) (siehe Institutsseite) Modulbeauftragte oder Modulbeauftragter Verwendbarkeit des Moduls in anderen Studiengängen: B. Ed. Mathematik, B. Sc. Informatik, B. Sc. Physik Die Übung hat einen LP mehr als im Lehramtsstudiengang. Entsprechend erhalten die B.Sc. zusätzliche ergänzende Übungsaufgaben Sonstiges Modul LAG2: Lineare Algebra und Geometrie 2: [Modul-Kennnummer ] Linear Algebra and Geometry 2 Pflicht- oder Wahlpflichtmodul: Pflichtmodul Leistungspunkte (LP) und 9 LP = 270 h Arbeitsaufwand (workload) Moduldauer 1 Semester (laut Studienverlaufsplan) Lehrveranstaltungen/ Lernformen: Art: Regelsemester b

In [ ]:
from pydantic import BaseModel

class Module(BaseModel):
    module_title: str
    ects: int
    optional: bool
    competencies: str
    content: str
    requirements: list[str] | None

# 1️⃣ Titel extrahieren
title_match = re.search(r'Modul [A-Z0-9]+: (.+?):', p0)
module_title = title_match.group(1) if title_match else "Unknown"

# 2️⃣ ECTS extrahieren
ects_match = re.search(r'Leistungspunkte.*?(\d+)\s*LP', p0)
ects = int(ects_match.group(1)) if ects_match else 0

# 3️⃣ Pflicht/Wahlpflicht
optional = "Pflichtmodul" not in p0

# 4️⃣ Requirements extrahieren
req_match = re.search(r'Empfohlene Teilnahmevoraussetzung\(en\) für das Modul: (.+?)\.', p0)
requirements = [req_match.group(1)] if req_match else []

# 5️⃣ Competencies extrahieren
comp_match = re.search(r'Qualifikationsziele/Lernergebnisse/Kompetenzen (.+)', p0, re.DOTALL)
competencies = comp_match.group(1).strip() if comp_match else ""

# 6️⃣ Content extrahieren (alles zwischen title und competencies)
content_match = re.search(rf'{module_title}.*?Qualifikationsziele/Lernergebnisse/Kompetenzen', p0, re.DOTALL)
content = content_match.group(0).strip() if content_match else ""

# 7️⃣ Modul erstellen
module = Module(
    module_title=module_title,
    ects=ects,
    optional=optional,
    competencies=competencies,
    content=content,
    requirements=requirements
)

print(module.model_dump_json(indent=2))

{
  "module_title": "Lineare Algebra und Geometrie 2",
  "ects": 9,
  "optional": false,
  "competencies": "In dieser Lehrveranstaltung erwerben die Studierenden ein grundlegendes Verständnis der Begriffe, Konzepte und Methoden der Linearen Algebra. Nach erfolgreichem Absolvieren des Moduls können Studierende in diesem Bereich - Definitionen, Begriffe, Aussagen und einfache Beweise wiedergeben. - mathematische Strukturen erkennen und benennen, und geeignete Notation verwenden und entwickeln. - Rechenverfahren routiniert durchführen und das Resultat durch Kontrollverfahren überprüfen. - bekannte Lösungsverfahren auf Problemstellungen anwenden und auf ähnliche Fragestellungen übertragen. - mathematische Hilfsmittel und Werkzeuge sachgerecht auswählen und flexibel einsetzen - Lösungsstrategien zu Problemstellungen entwickeln und umsetzen, und heuristische Methoden zum Problemlösen anwenden - sicher und kritisch über die Themen der linearen Algebra in mathematischer Fachsprache präzise kom